# 01 · Data Exploration

Explore the **canonical tables** produced by the ingest stage — the grid (with
Oregon ecoregions and static features), the weather/vegetation panel, and the
fire-event records (with ignition cause).

> Run the ingest stage first, e.g. `uv run python scripts/01_ingest.py --synthetic`.
> Everything here is source-agnostic: it works the same on real Earth Engine data.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from wildfire.config import load_config
from wildfire.ingest.datasets import load_canonical

cfg = load_config()
data = load_canonical(cfg)
grid, panel, events = data['grid'], data['weather_panel'], data['fire_events']
panel['date'] = pd.to_datetime(panel['date'])
print('source:', data['manifest'].get('source'))
print(f"grid cells: {len(grid):,} | panel rows: {len(panel):,} | events: {len(events):,}")
print(f"positive (fire) rate: {panel['fire'].mean()*100:.2f}%  <- rare-event problem")

## The grid & Oregon ecoregions

The unified H3 grid, colored by ecoregion — the natural region unit for the
fire-count model and a strong spatial-blocking variable for cross-validation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
grid.plot(column='ecoregion', legend=True, ax=axes[0], cmap='tab10',
          legend_kwds={'fontsize': 7, 'loc': 'lower left'})
axes[0].set_title('Oregon ecoregions'); axes[0].set_axis_off()
if 'elevation' in grid:
    grid.plot(column='elevation', legend=True, ax=axes[1], cmap='terrain')
    axes[1].set_title('Elevation proxy (m)'); axes[1].set_axis_off()
plt.tight_layout()

## Fire events: timing and ignition cause

Ignition cause (lightning vs human) is the signal most models underuse — note how
the mix differs and how fires concentrate in the warm/dry season.

In [ ]:
events['date'] = pd.to_datetime(events['date'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
events.groupby(events['date'].dt.month).size().reindex(range(1,13), fill_value=0).plot(
    kind='bar', ax=axes[0], color='#c1440e')
axes[0].set_title('Fires by month'); axes[0].set_xlabel('month')
if 'cause' in events:
    events['cause'].value_counts().plot(kind='bar', ax=axes[1], color=['#444','#f4a300','#888'])
    axes[1].set_title('Ignition cause')
if 'size_ha' in events:
    np.log10(events['size_ha'].clip(lower=0.1)).hist(bins=40, ax=axes[2], color='#7a3b2e')
    axes[2].set_title('Fire size (log10 ha)')
plt.tight_layout()

## Where do lightning vs human fires occur?

Spatial cause geography: human ignitions cluster near infrastructure/valleys,
lightning in the mountains/east — exactly the prior the risk model exploits.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
grid.boundary.plot(ax=ax, color='0.85', linewidth=0.2)
for cause, color in [('lightning', '#f4a300'), ('human', '#1f6feb')]:
    sub = events[events.get('cause') == cause]
    if len(sub):
        ax.scatter(sub['lon'], sub['lat'], s=3, alpha=0.3, label=cause, color=color)
ax.legend(); ax.set_title('Fire ignitions by cause'); ax.set_axis_off()

## Weather drivers by ecoregion

VPD (vapor pressure deficit) and ERC (energy release component) are the
fire-danger signals; note the west-wet / east-dry contrast.

In [ ]:
pe = panel.merge(grid[['cell_id', 'ecoregion']], on='cell_id', how='left')
order = pe.groupby('ecoregion')['vpd'].median().sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, col in zip(axes, ['vpd', 'erc']):
    pe.boxplot(column=col, by='ecoregion', ax=ax, rot=90, grid=False)
    ax.set_title(col.upper()); ax.set_xlabel('')
plt.suptitle(''); plt.tight_layout()

**Next:** `02_features_and_imbalance.ipynb` — engineered features and the
class-imbalance strategy.